# WESAD 전체(16명) 상태별 생체신호 변화 분석

이 노트북은 WESAD 16명(`S2`~`S17`, `S12` 제외) 데이터를 모두 읽어,
- 상태(`Baseline=1`, `Stress=2`, `Amusement=3`)별
- 신호(`BVP`, `ACC`, `Chest EMG`, `EDA`, `TEMP`)의
평균과 표준편차를 계산하고 시각화합니다.

> 참고: WESAD는 센서별 샘플링레이트가 다르므로, 각 신호 길이에 맞춰 라벨을 시간 비율 기준으로 정렬합니다.

In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

WESAD_ROOT = Path(r"C:\Users\user\Desktop\Myproject\Dataset\WESAD")
STATE_MAP = {1: "Baseline", 2: "Stress", 3: "Amusement"}

print(f"WESAD 경로: {WESAD_ROOT}")
print(f"존재 여부: {WESAD_ROOT.exists()}")

In [ ]:
def align_labels_to_signal(signal_len: int, labels_700hz: np.ndarray) -> np.ndarray:
    """
    700Hz 라벨 벡터를 임의 길이의 신호 길이에 맞춰 시간 비율로 정렬.
    """
    idx = (np.arange(signal_len) * (len(labels_700hz) / signal_len)).astype(int)
    idx = np.clip(idx, 0, len(labels_700hz) - 1)
    return labels_700hz[idx]


def subject_feature_stats(subject_pkl_path: Path) -> pd.DataFrame:
    """
    단일 subject에서 상태별 feature 평균을 반환.
    반환 컬럼: [subject, state, BVP, ACC, Chest_EMG, EDA, TEMP]
    """
    with open(subject_pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    labels = data["label"].reshape(-1)
    chest = data["signal"]["chest"]
    wrist = data["signal"]["wrist"]

    features = {
        "BVP": wrist["BVP"].reshape(-1),
        "ACC": np.linalg.norm(wrist["ACC"], axis=1),
        "Chest_EMG": chest["EMG"].reshape(-1),
        "EDA": chest["EDA"].reshape(-1),
        "TEMP": chest["Temp"].reshape(-1),
    }

    rows = []
    subject_id = subject_pkl_path.stem

    for state_id, state_name in STATE_MAP.items():
        row = {"subject": subject_id, "state": state_name}
        valid_state = True

        for feat_name, feat_arr in features.items():
            aligned_labels = align_labels_to_signal(len(feat_arr), labels)
            mask = aligned_labels == state_id

            if np.any(mask):
                row[feat_name] = float(np.mean(feat_arr[mask]))
            else:
                row[feat_name] = np.nan
                valid_state = False

        if valid_state:
            rows.append(row)

    return pd.DataFrame(rows)


subject_dirs = sorted([p for p in WESAD_ROOT.glob("S*") if p.is_dir()])
subject_pkls = [p / f"{p.name}.pkl" for p in subject_dirs if (p / f"{p.name}.pkl").exists()]

actual_subjects = [p.stem for p in subject_pkls]
expected_subjects = [f"S{i}" for i in range(2, 18)]
missing_subjects = sorted(set(expected_subjects) - set(actual_subjects))

print(f"탐지된 Subject 수: {len(subject_pkls)}")
print("Subjects:", actual_subjects)
if missing_subjects:
    print("누락된 Subject:", missing_subjects)
    print("※ 공개 WESAD 기본 구성은 보통 S12 제외(총 15명)입니다.")

In [ ]:
# 16명 전체 subject에 대해 상태별 feature 평균 계산
subject_level_rows = []
for pkl_path in subject_pkls:
    subject_df = subject_feature_stats(pkl_path)
    subject_level_rows.append(subject_df)

subject_state_df = pd.concat(subject_level_rows, ignore_index=True)

# 결측 제거(안전장치)
subject_state_df = subject_state_df.dropna().reset_index(drop=True)

print("subject_state_df shape:", subject_state_df.shape)
display(subject_state_df.head())

group_count = subject_state_df.groupby("state")["subject"].nunique().rename("n_subjects")
print("\n상태별 포함 subject 수")
display(group_count)

In [ ]:
# 상태별(섹션별) 전체 통계: mean ± std
features = ["BVP", "ACC", "Chest_EMG", "EDA", "TEMP"]
summary = (
    subject_state_df
    .groupby("state")[features]
    .agg(["mean", "std"])
)

# 보기 쉽게 평탄화
summary_flat = summary.copy()
summary_flat.columns = [f"{feat}_{stat}" for feat, stat in summary_flat.columns]
summary_flat = summary_flat.reset_index()

print("상태별 평균/표준편차 요약")
display(summary_flat)

In [ ]:
# Fig 1) 전체적인 판단 기준: 상태별 feature z-score heatmap
state_mean = subject_state_df.groupby("state")[features].mean()
state_z = (state_mean - state_mean.mean(axis=0)) / state_mean.std(axis=0)

plt.figure(figsize=(9, 4.8))
sns.heatmap(
    state_z,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    cbar_kws={"label": "Feature z-score"}
)
plt.title("WESAD 상태별 상대적 변화(전체 판단 기준)")
plt.xlabel("Signal Feature")
plt.ylabel("State")
plt.tight_layout()
plt.show()

In [ ]:
# Fig 2) 상태별 평균 ± 표준편차 Bar graph
plot_df = subject_state_df.melt(
    id_vars=["subject", "state"],
    value_vars=features,
    var_name="feature",
    value_name="value"
)

plt.figure(figsize=(12, 5.6))
ax = sns.barplot(
    data=plot_df,
    x="feature",
    y="value",
    hue="state",
    errorbar="sd",
    capsize=0.08
)
plt.title("상태별 생체신호 평균 ± 표준편차 (16명)")
plt.xlabel("Feature")
plt.ylabel("Signal Value")
plt.legend(title="State", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 해석 가이드

- Heatmap은 각 feature에서 상태별 평균을 z-score로 표준화한 값입니다. 
  - 0보다 크면 해당 상태에서 상대적으로 높은 편,
  - 0보다 작으면 상대적으로 낮은 편을 의미합니다.
- Bar graph는 상태별 평균과 표준편차(개인 간 변동)를 동시에 보여줍니다.
- 일반적으로 WESAD에서는 `Stress`에서 EDA/EMG가 증가하고, `Amusement`는 Baseline 대비 일부 각성 신호가 중간 수준으로 나타나는 경향을 확인할 수 있습니다.
- `ACC`는 3축 벡터 크기(norm)로 계산되어 움직임 크기의 대표값으로 해석할 수 있습니다.